# 04 · Fitur possession (H vs A)

**Tujuan.** Menghitung, untuk setiap frame, bukti tentang **tim mana yang menguasai bola** — dalam dua keluarga:

1. **Bukti bola** (hanya saat bola terlihat): siapa yang terdekat, seberapa jauh, berapa pemain menekan.
2. **Bukti bentuk tim** (selalu tersedia): lebar, kedalaman, sebaran, kecepatan, gerak ke gawang lawan. Temuan
   `02` #4: tim yang menguasai bola lebih **lebar**, lebih **menyebar**, dan lebih **lambat** — dan sinyal ini tetap ada
   saat bola hilang.

Notebook ini **tidak** mengambil keputusan; ia hanya menyiapkan fitur dan mengukur cakupannya. Model ada di `05`.

| | |
|---|---|
| **Input** | `cache/prep.npz` |
| **Output** | `cache/features_ha.npz`, laporan `cache/reports/04_fitur_possession.*` |
| **Waktu** | ± 30 s |

### Konvensi

- Setiap fitur tim dihitung untuk H dan A, lalu diambil **selisih H − A**: nilai positif = bukti untuk `H`
  (kecuali yang tandanya terbalik secara alami, seperti kecepatan; tanda tidak diubah di sini — modellah yang belajar
  arah dan besarnya).
- "Ke depan" = ke arah gawang lawan tim itu (`attack_dir` dari `01`, terverifikasi).
- Fitur bentuk tim = `NaN` bila tim itu terlacak < `MIN_PLAYERS` slot (bentuknya tak bisa dipercaya).
- Versi halus = rata-rata bergerak terpusat ± `SMOOTH_S` detik: bentuk tim berubah lambat, per frame berisik.

In [ ]:
%run ./common.ipynb
import warnings

## Parameter

| Parameter | Nilai | Alasan |
|---|---|---|
| `MIN_PLAYERS` | 6 | di bawah ini rentang/sebaran tim didominasi pemain yang hilang |
| `SMOOTH_S` | 2,0 s | skala perubahan bentuk tim; sama dengan analisis `02` |
| `VEL_HALF` | 10 frame | kecepatan per slot dari selisih posisi ± 0,4 s |
| `VEL_CLIP` | 12 m/s | batas kecepatan pemain yang masuk akal; di atasnya hampir pasti lompatan identitas slot |
| `PRESS_R` | 3,0 m | radius "menekan bola" |
| `NEAR_SHARE_WIN` | 51 frame | jendela (± 1 s) porsi frame H terdekat |

In [ ]:
MIN_PLAYERS = 6
SMOOTH_S = 2.0
VEL_HALF = 10
VEL_CLIP = 12.0
PRESS_R = 3.0
NEAR_SHARE_WIN = 51

In [ ]:
rep = Report("04_fitur_possession")
z = load_prep()
P, HOME = z["players"], z["slot_is_home"]
obs = z["ball_src"] > 0
bx, by = z["ball_filled"][:, 0], z["ball_filled"][:, 1]
adir = z["attack_dir"].astype(np.float32)
SMOOTH = int(SMOOTH_S * FPS) * 2 + 1
FEAT = {}

## 1. Bukti bola

| Fitur | Definisi |
|---|---|
| `d_H`, `d_A` | jarak pemain terdekat tiap tim ke bola (m) |
| `near` | +1 bila H terdekat, −1 bila A, 0 bila bola hilang |
| `near_margin` | `d_A − d_H`, dibatasi ±15 m — seberapa jelas keunggulannya |
| `near_share` | porsi frame H-terdekat dikurangi A-terdekat dalam ± 1 s (dari frame terlihat), ∈ [−1, 1] |
| `press_H`, `press_A` | jumlah pemain tiap tim dalam `PRESS_R` dari bola |
| `ball_v` | kecepatan bola (m/s) dari selisih posisi ± 6 frame |

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    DH = np.hypot(P[:, HOME, 0] - bx[:, None], P[:, HOME, 1] - by[:, None])
    DA = np.hypot(P[:, ~HOME, 0] - bx[:, None], P[:, ~HOME, 1] - by[:, None])
    d_H = np.where(obs, np.nanmin(DH, axis=1), np.nan)
    d_A = np.where(obs, np.nanmin(DA, axis=1), np.nan)
    press_H = np.where(obs, (DH < PRESS_R).sum(1), np.nan)
    press_A = np.where(obs, (DA < PRESS_R).sum(1), np.nan)

vis = np.isfinite(d_H) & np.isfinite(d_A)
near = np.where(vis, np.where(d_H < d_A, 1, -1), 0).astype(np.int8)
s_sum = pd.Series(near.astype(float)).rolling(NEAR_SHARE_WIN, center=True, min_periods=1).sum()
s_cnt = pd.Series(vis.astype(float)).rolling(NEAR_SHARE_WIN, center=True, min_periods=1).sum()
near_share = np.where(s_cnt >= 5, s_sum / s_cnt.clip(lower=1), np.nan)

k = 6
ball_v = np.full(N_FRAMES, np.nan)
ball_v[k:-k] = np.hypot(bx[2 * k:] - bx[:-2 * k], by[2 * k:] - by[:-2 * k]) / (2 * k / FPS)

FEAT.update(d_H=d_H, d_A=d_A, near=near, near_margin=np.clip(d_A - d_H, -15, 15), near_share=near_share,
            press_H=press_H, press_A=press_A, ball_v=ball_v)
rep("bola terlihat dengan kedua tim terlacak: {:.1f}% frame dinilai".format(100 * vis[GRADED].mean()))

## 2. Bukti bentuk tim

| Fitur per tim | Definisi | Temuan `02` |
|---|---|---|
| `width` | rentang y (max − min) | tim menguasai lebih lebar (AUC selisih ± 0,95) |
| `depth` | rentang x | lebih panjang (± 0,77) |
| `spread` | √(var x + var y) | lebih menyebar (± 0,91) |
| `advance` | sentroid x relatif garis tengah, ke depan | tak informatif (± 0,48) — tetap disimpan, model yang memutuskan |
| `speed` | kecepatan rata-rata pemain (kolom ATD) | tim menguasai lebih **lambat** (± 0,14 → terbalik) |
| `v_goal` | rata-rata kecepatan ke depan per slot (± 0,4 s, dibatasi `VEL_CLIP`) | lebih maju bergerak (± 0,65) |

In [ ]:
def team_shape(mask, sign, n_tracked):
    X, Y, S = P[:, mask, 0], P[:, mask, 1], P[:, mask, 2]
    ok = n_tracked >= MIN_PLAYERS
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore", category=RuntimeWarning)
        vx = np.full(X.shape, np.nan, dtype=np.float32)
        vx[VEL_HALF:-VEL_HALF] = (X[2 * VEL_HALF:] - X[:-2 * VEL_HALF]) / (2 * VEL_HALF / FPS)
        vx[np.abs(vx) > VEL_CLIP] = np.nan
        out = {
            "width": np.nanmax(Y, axis=1) - np.nanmin(Y, axis=1),
            "depth": np.nanmax(X, axis=1) - np.nanmin(X, axis=1),
            "spread": np.sqrt(np.nanvar(X, axis=1) + np.nanvar(Y, axis=1)),
            "advance": (np.nanmean(X, axis=1) - PITCH_L / 2) * sign,
            "speed": np.nanmean(S, axis=1),
            "v_goal": np.nanmean(vx, axis=1) * sign,
        }
    return {k: np.where(ok, v, np.nan).astype(np.float32) for k, v in out.items()}


def smooth(v):
    return pd.Series(v).rolling(SMOOTH, center=True, min_periods=SMOOTH // 4).mean().to_numpy().astype(np.float32)


with timer("bentuk tim"):
    sH = team_shape(HOME, adir, z["n_home"])
    sA = team_shape(~HOME, -adir, z["n_away"])

SHAPE = ["width", "depth", "spread", "advance", "speed", "v_goal"]
for name in SHAPE:
    FEAT[name + "_H"], FEAT[name + "_A"] = sH[name], sA[name]
    diff = sH[name] - sA[name]
    FEAT[name + "_diff"] = diff.astype(np.float32)
    FEAT[name + "_diff_s"] = smooth(diff)
FEAT["n_home"], FEAT["n_away"] = z["n_home"], z["n_away"]
rep("bentuk tim tersedia (kedua tim ≥ {} slot): {:.1f}% frame dinilai".format(
    MIN_PLAYERS, 100 * np.isfinite(FEAT["width_diff"])[GRADED].mean()))

## 3. Cakupan & ringkasan fitur

Untuk setiap fitur: cakupan (% frame dinilai yang terukur), median, dan rentang p5–p95. Ini memastikan tidak ada
fitur yang hampir selalu `NaN` atau bernilai aneh sebelum dipakai model.

In [ ]:
rows = []
for name, v in FEAT.items():
    g = np.asarray(v, dtype=float)[GRADED]
    fin = np.isfinite(g)
    rows.append({"fitur": name, "cakupan %": round(100 * fin.mean(), 1),
                 "median": round(float(np.median(g[fin])), 2) if fin.any() else np.nan,
                 "p5": round(float(np.percentile(g[fin], 5)), 2) if fin.any() else np.nan,
                 "p95": round(float(np.percentile(g[fin], 95)), 2) if fin.any() else np.nan})
cover = pd.DataFrame(rows).set_index("fitur")
rep.table(cover, "cakupan fitur (frame dinilai):")

### Korelasi antar fitur selisih (halus)

Fitur yang sangat berkorelasi membawa informasi yang sama; model linear di `05` memakai regularisasi supaya
tidak goyah karenanya, tetapi bagus untuk diketahui.

In [ ]:
diff_s = [n + "_diff_s" for n in SHAPE]
M = np.stack([FEAT[n] for n in diff_s], axis=1)[GRADED]
M = M[np.isfinite(M).all(1)]
corr = pd.DataFrame(np.corrcoef(M.T), index=SHAPE, columns=SHAPE).round(2)
rep.table(corr, "korelasi selisih bentuk tim (± {} s):".format(SMOOTH_S))

### Informasi saja: daya beda di GT publik

AUC untuk `H` vs `A` di frame hidup jendela publik — hanya untuk memastikan fitur yang **dihitung ulang** di sini
berperilaku sama dengan analisis `02`. Tidak dipakai untuk memilih fitur atau parameter.

In [ ]:
gt = pm.read_label_csv(str(GT_PUBLIC))["label"].astype(str).str.strip().str.upper().to_numpy()
isH, isA = gt == "H", gt == "A"
auc_pub = pd.DataFrame([{"fitur": n, "AUC H vs A": round(auc(np.asarray(FEAT[n], float)[isH],
                                                           np.asarray(FEAT[n], float)[isA]), 3)}
                        for n in ["near_margin", "near_share"] + diff_s]).set_index("fitur")
rep.table(auc_pub, "AUC publik (informasi):")

## 4. Simpan

In [ ]:
names = list(FEAT)
np.savez(CACHE / "features_ha.npz", names=np.array(names),
         **{n: np.asarray(v).astype(np.float32 if np.asarray(v).dtype.kind == "f" else np.asarray(v).dtype)
            for n, v in FEAT.items()})
with np.load(CACHE / "features_ha.npz") as fz:
    assert all(fz[n].shape == (N_FRAMES,) for n in names)
rep("tersimpan: {} fitur → cache/features_ha.npz".format(len(names)))
rep.save({"features": names, "coverage": cover.reset_index().to_dict("records"),
          "corr": corr.to_dict(), "auc_public": auc_pub.reset_index().to_dict("records")})

## Ringkasan & keputusan

- **Cakupan:** bukti bola tersedia di ± 59% frame dinilai; **bukti bentuk tim di ± 94%** (kedua tim ≥ 6 slot). Bentuk
  tim adalah satu-satunya bukti untuk sebagian besar frame tanpa bola.
- **Konsistensi:** AUC publik fitur yang dihitung ulang sama dengan analisis `02` (lebar ± 0,95, sebaran ± 0,92,
  kecepatan ± 0,14 → terbalik), jadi implementasinya benar.
- **Korelasi:** kedalaman, sebaran, dan majunya tim saling berkorelasi tinggi (0,8–0,9); lebar dan kecepatan membawa
  informasi tersendiri. → model di `05` memakai regularisasi L2 dan koefisiennya dibaca sebagai satu kesatuan,
  bukan satu per satu.

**Berikutnya:** `05_model_possession.ipynb`.